In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
from pathlib import Path

TEAM_ROOT = Path("/content/drive/MyDrive/TeamProject")

print(f"[ROOT] {TEAM_ROOT}\n")

for item in sorted(TEAM_ROOT.iterdir(), key=lambda p: p.name.lower()):
    if item.is_dir():
        print(f"[DIR]  {item.name}")

In [ ]:
import os
from pathlib import Path

TEST_ROOT = Path("/content/drive/MyDrive/TeamProject/test_dataset")
MAX_DEPTH = 2

print(f"[ROOT] {TEST_ROOT}\n")

for current_path, dirnames, filenames in os.walk(TEST_ROOT):
    current = Path(current_path)
    relative = current.relative_to(TEST_ROOT)
    depth = len(relative.parts)

    if depth > MAX_DEPTH:
        dirnames[:] = []
        continue

    indent = "  " * depth
    folder_name = TEST_ROOT.name if depth == 0 else current.name
    print(f"{indent}[DIR] {folder_name}")

    # runs 내부 파일은 다음 코드에서 따로 확인
    if current.name != "runs":
        for filename in sorted(filenames):
            print(f"{indent}  [FILE] {filename}")

In [ ]:
import os
from pathlib import Path

RUNS_ROOT = Path("/content/drive/MyDrive/TeamProject/test_dataset/runs")
OUTPUT_FILE = Path("/content/runs_structure.txt")

lines = [f"[ROOT] {RUNS_ROOT}", ""]

if not RUNS_ROOT.exists():
    raise FileNotFoundError(f"runs 폴더를 찾을 수 없습니다: {RUNS_ROOT}")

for current_path, dirnames, filenames in os.walk(RUNS_ROOT):
    dirnames.sort()
    filenames.sort()

    current = Path(current_path)
    relative = current.relative_to(RUNS_ROOT)
    depth = len(relative.parts)
    indent = "  " * depth

    folder_name = RUNS_ROOT.name if depth == 0 else current.name
    lines.append(f"{indent}[DIR] {folder_name}")

    for filename in filenames:
        file_path = current / filename

        try:
            size_mb = file_path.stat().st_size / (1024 * 1024)
            lines.append(
                f"{indent}  [FILE] {filename} ({size_mb:.2f} MB)"
            )
        except OSError:
            lines.append(f"{indent}  [FILE] {filename}")

OUTPUT_FILE.write_text("\n".join(lines), encoding="utf-8")

print("\n".join(lines))
print(f"\n저장 완료: {OUTPUT_FILE}")

In [ ]:
from pathlib import Path

TEST_ROOT = Path(
    "/content/drive/MyDrive/TeamProject/test_dataset/실사용테스트"
)

OLD_IMAGES = TEST_ROOT / "imagies"
NEW_IMAGES = TEST_ROOT / "images"

if OLD_IMAGES.exists() and not NEW_IMAGES.exists():
    OLD_IMAGES.rename(NEW_IMAGES)
    print(f"변경 완료: {OLD_IMAGES.name} → {NEW_IMAGES.name}")
elif NEW_IMAGES.exists():
    print(f"이미 정상입니다: {NEW_IMAGES}")
else:
    print("imagies 또는 images 폴더를 찾지 못했습니다.")

In [ ]:
ONE_STAGE_WEIGHT = (
    "/content/drive/MyDrive/TeamProject/test_dataset/runs/"
    "yolov8n_1stage_9class_bboxfixed/weights/best.pt"
)

TWO_STAGE_DETECTOR_WEIGHT = (
    "/content/drive/MyDrive/TeamProject/test_dataset/runs/"
    "yolov8n_2stage_material3_bboxfixed/weights/best.pt"
)

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path("/content/drive/MyDrive/TeamProject/test_dataset")
RUNS = ROOT / "runs"

def show_json(path):
    path = Path(path)
    print("\n" + "=" * 90)
    print(f"[JSON] {path.relative_to(ROOT)}")

    if not path.exists():
        print("파일 없음")
        return

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    print(json.dumps(data, ensure_ascii=False, indent=2))


def show_csv(path, tail=None):
    path = Path(path)
    print("\n" + "=" * 90)
    print(f"[CSV] {path.relative_to(ROOT)}")

    if not path.exists():
        print("파일 없음")
        return

    df = pd.read_csv(path)

    if tail is not None:
        print(df.tail(tail).to_string(index=False))
    else:
        print(df.to_string(index=False))


def show_text(path):
    path = Path(path)
    print("\n" + "=" * 90)
    print(f"[TEXT] {path.relative_to(ROOT)}")

    if not path.exists():
        print("파일 없음")
        return

    print(path.read_text(encoding="utf-8", errors="replace"))

In [ ]:
# 데이터 경로와 클래스 구성
show_text(ROOT / "1-stage" / "data.yaml")
show_text(ROOT / "2-stage" / "data.yaml")

# YOLO 학습 설정
show_text(
    RUNS / "yolov8n_1stage_9class_bboxfixed" / "args.yaml"
)
show_text(
    RUNS / "yolov8n_2stage_material3_bboxfixed" / "args.yaml"
)

# 1-stage 별도 best 모델 평가 요약
show_json(
    RUNS
    / "yolov8n_1stage_9class_bboxfixed"
    / "best_model_evaluation_summary.json"
)

# ResNet padding 비교
show_csv(RUNS / "resnet18_padding_comparison.csv")

for pad in ["pad000", "pad005", "pad010"]:
    show_json(
        RUNS
        / f"resnet18_dirty3_bboxfixed_{pad}"
        / "best_metrics.json"
    )

# 기존 end-to-end 결과
E2E = RUNS / "end_to_end_bboxfixed_validation"

show_json(E2E / "final_summary.json")
show_csv(E2E / "end_to_end_comparison.csv")
show_csv(E2E / "best_threshold_summary.csv")
show_csv(E2E / "common_threshold_025_summary.csv")
show_csv(E2E / "threshold_sweep.csv")

# 학습 마지막 epoch 지표
show_csv(
    RUNS / "yolov8n_1stage_9class_bboxfixed" / "results.csv",
    tail=5
)
show_csv(
    RUNS / "yolov8n_2stage_material3_bboxfixed" / "results.csv",
    tail=5
)

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA 사용 가능:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/TeamProject/test_dataset")
RUNS = ROOT / "runs"

ONE_STAGE_WEIGHT = (
    RUNS
    / "yolov8n_1stage_9class_bboxfixed"
    / "weights"
    / "best.pt"
)

TWO_STAGE_DETECTOR_WEIGHT = (
    RUNS
    / "yolov8n_2stage_material3_bboxfixed"
    / "weights"
    / "best.pt"
)

EXTERNAL_ROOT = ROOT / "실사용테스트"
EXTERNAL_IMAGES = EXTERNAL_ROOT / "images"
EXTERNAL_LABELS = EXTERNAL_ROOT / "labels"

paths = {
    "1-stage weight": ONE_STAGE_WEIGHT,
    "2-stage detector weight": TWO_STAGE_DETECTOR_WEIGHT,
    "external images": EXTERNAL_IMAGES,
    "external labels": EXTERNAL_LABELS,
    "1-stage data.yaml": ROOT / "1-stage" / "data.yaml",
    "2-stage data.yaml": ROOT / "2-stage" / "data.yaml",
}

print("[중요 경로 검사]\n")

for name, path in paths.items():
    status = "OK" if path.exists() else "MISSING"
    print(f"{name:27} | {status}")
    print(f"  {path}")

In [ ]:
from pathlib import Path
from collections import Counter

ROOT = Path("/content/drive/MyDrive/TeamProject/test_dataset")

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def audit_split(dataset_root, split="val"):
    dataset_root = Path(dataset_root)
    image_dir = dataset_root / "images" / split
    label_dir = dataset_root / "labels" / split

    images = {
        p.stem: p
        for p in image_dir.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    }

    labels = {
        p.stem: p
        for p in label_dir.glob("*.txt")
    }

    class_counts = Counter()
    object_count = 0
    malformed = []

    for stem, label_path in labels.items():
        lines = [
            line.strip()
            for line in label_path.read_text(
                encoding="utf-8",
                errors="replace"
            ).splitlines()
            if line.strip()
        ]

        for line_number, line in enumerate(lines, start=1):
            parts = line.split()

            if len(parts) != 5:
                malformed.append(
                    (label_path.name, line_number, line)
                )
                continue

            try:
                class_id = int(parts[0])
                coords = list(map(float, parts[1:]))
            except ValueError:
                malformed.append(
                    (label_path.name, line_number, line)
                )
                continue

            if any(value < 0 or value > 1 for value in coords):
                malformed.append(
                    (label_path.name, line_number, line)
                )
                continue

            class_counts[class_id] += 1
            object_count += 1

    print("=" * 80)
    print(f"Dataset: {dataset_root.name}")
    print(f"Image directory: {image_dir}")
    print(f"Label directory: {label_dir}")
    print(f"Images: {len(images)}")
    print(f"Labels: {len(labels)}")
    print(f"Objects: {object_count}")
    print(f"Missing labels: {len(images.keys() - labels.keys())}")
    print(f"Missing images: {len(labels.keys() - images.keys())}")
    print(f"Malformed annotations: {len(malformed)}")
    print(f"Class counts: {dict(sorted(class_counts.items()))}")

    if images.keys() - labels.keys():
        print(
            "Missing label examples:",
            sorted(images.keys() - labels.keys())[:10]
        )

    if labels.keys() - images.keys():
        print(
            "Missing image examples:",
            sorted(labels.keys() - images.keys())[:10]
        )

    if malformed:
        print("Malformed examples:", malformed[:10])


audit_split(ROOT / "1-stage", "val")
audit_split(ROOT / "2-stage", "val")

In [ ]:
def audit_external(image_dir, label_dir):
    image_dir = Path(image_dir)
    label_dir = Path(label_dir)

    images = {
        p.stem: p
        for p in image_dir.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    }
    labels = {
        p.stem: p
        for p in label_dir.glob("*.txt")
    }

    class_counts = Counter()
    line_counts = Counter()
    malformed = []

    for stem, label_path in labels.items():
        lines = [
            line.strip()
            for line in label_path.read_text(
                encoding="utf-8",
                errors="replace"
            ).splitlines()
            if line.strip()
        ]

        line_counts[len(lines)] += 1

        for line_number, line in enumerate(lines, start=1):
            parts = line.split()

            try:
                class_id = int(parts[0])
                coords = list(map(float, parts[1:]))
            except (ValueError, IndexError):
                malformed.append(
                    (label_path.name, line_number, line)
                )
                continue

            if (
                len(parts) != 5
                or class_id < 0
                or class_id > 8
                or any(value < 0 or value > 1 for value in coords)
            ):
                malformed.append(
                    (label_path.name, line_number, line)
                )
                continue

            class_counts[class_id] += 1

    print("[실사용 데이터 감사]")
    print("Images:", len(images))
    print("Labels:", len(labels))
    print("GT objects:", sum(class_counts.values()))
    print("Images by object count:", dict(sorted(line_counts.items())))
    print("Class counts:", dict(sorted(class_counts.items())))
    print("Missing labels:", sorted(images.keys() - labels.keys()))
    print("Missing images:", sorted(labels.keys() - images.keys()))
    print("Malformed:", malformed)


audit_external(EXTERNAL_IMAGES, EXTERNAL_LABELS)

In [ ]:
%pip install -q ultralytics

In [ ]:
try:
    import ultralytics
    print("Ultralytics version:", ultralytics.__version__)
except ImportError:
    print("Ultralytics가 설치되어 있지 않습니다.")

In [ ]:
from pathlib import Path
from ultralytics import YOLO
import ultralytics
import torch
import json

ROOT = Path("/content/drive/MyDrive/TeamProject/test_dataset")
RUNS = ROOT / "runs"

ONE_STAGE_WEIGHT = (
    RUNS
    / "yolov8n_1stage_9class_bboxfixed"
    / "weights"
    / "best.pt"
)

TWO_STAGE_DETECTOR_WEIGHT = (
    RUNS
    / "yolov8n_2stage_material3_bboxfixed"
    / "weights"
    / "best.pt"
)

ONE_STAGE_DATA = ROOT / "1-stage" / "data.yaml"
TWO_STAGE_DATA = ROOT / "2-stage" / "data.yaml"

REVAL_DIR = RUNS / "standalone_revalidation"

print("Ultralytics:", ultralytics.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

for path in [
    ONE_STAGE_WEIGHT,
    TWO_STAGE_DETECTOR_WEIGHT,
    ONE_STAGE_DATA,
    TWO_STAGE_DATA,
]:
    if not path.exists():
        raise FileNotFoundError(path)

print("경로 검사 완료")

In [ ]:
one_model = YOLO(str(ONE_STAGE_WEIGHT))

one_metrics = one_model.val(
    data=str(ONE_STAGE_DATA),
    split="val",
    imgsz=640,
    batch=32,
    device=0,
    workers=2,
    plots=True,
    project=str(REVAL_DIR),
    name="one_stage_best",
    exist_ok=True,
    verbose=True,
)

one_summary = {
    "model": str(ONE_STAGE_WEIGHT),
    "data": str(ONE_STAGE_DATA),
    "ultralytics_version": ultralytics.__version__,
    "images": 3158,
    "precision": float(one_metrics.box.mp),
    "recall": float(one_metrics.box.mr),
    "mAP50": float(one_metrics.box.map50),
    "mAP75": float(one_metrics.box.map75),
    "mAP50_95": float(one_metrics.box.map),
    "speed_ms_per_image": {
        key: float(value)
        for key, value in one_metrics.speed.items()
    },
    "per_class_mAP50_95": {
        one_model.names[index]: float(value)
        for index, value in enumerate(one_metrics.box.maps)
    },
}

with open(
    REVAL_DIR / "one_stage_best_summary.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(one_summary, f, ensure_ascii=False, indent=2)

print(json.dumps(one_summary, ensure_ascii=False, indent=2))

In [ ]:
two_detector = YOLO(str(TWO_STAGE_DETECTOR_WEIGHT))

two_metrics = two_detector.val(
    data=str(TWO_STAGE_DATA),
    split="val",
    imgsz=640,
    batch=32,
    device=0,
    workers=2,
    plots=True,
    project=str(REVAL_DIR),
    name="two_stage_detector_best",
    exist_ok=True,
    verbose=True,
)

two_summary = {
    "model": str(TWO_STAGE_DETECTOR_WEIGHT),
    "data": str(TWO_STAGE_DATA),
    "ultralytics_version": ultralytics.__version__,
    "images": 3158,
    "precision": float(two_metrics.box.mp),
    "recall": float(two_metrics.box.mr),
    "mAP50": float(two_metrics.box.map50),
    "mAP75": float(two_metrics.box.map75),
    "mAP50_95": float(two_metrics.box.map),
    "speed_ms_per_image": {
        key: float(value)
        for key, value in two_metrics.speed.items()
    },
    "per_class_mAP50_95": {
        two_detector.names[index]: float(value)
        for index, value in enumerate(two_metrics.box.maps)
    },
}

with open(
    REVAL_DIR / "two_stage_detector_best_summary.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(two_summary, f, ensure_ascii=False, indent=2)

print(json.dumps(two_summary, ensure_ascii=False, indent=2))

In [ ]:
from pathlib import Path
import torch
import json

ROOT = Path("/content/drive/MyDrive/TeamProject/test_dataset")
RUNS = ROOT / "runs"

CLASSIFIER_DIR = (
    RUNS / "resnet18_dirty3_bboxfixed_pad005"
)

FULL_CHECKPOINT = (
    CLASSIFIER_DIR / "best_resnet18_dirty3_checkpoint.pt"
)

STATE_DICT_FILE = (
    CLASSIFIER_DIR / "best_resnet18_dirty3_state_dict.pt"
)

print("[파일 확인]")
print("Full checkpoint:", FULL_CHECKPOINT.exists(), FULL_CHECKPOINT)
print("State dict:", STATE_DICT_FILE.exists(), STATE_DICT_FILE)

checkpoint = torch.load(
    FULL_CHECKPOINT,
    map_location="cpu",
    weights_only=False,
)

print("\n[Checkpoint 타입]")
print(type(checkpoint))

if isinstance(checkpoint, dict):
    print("\n[Checkpoint 최상위 키]")
    print(list(checkpoint.keys()))

    print("\n[Tensor가 아닌 설정값]")
    for key, value in checkpoint.items():
        if isinstance(value, torch.Tensor):
            print(f"{key}: Tensor {tuple(value.shape)}")
        elif isinstance(value, dict):
            print(
                f"{key}: dict, 항목 수={len(value)}, "
                f"앞쪽 키={list(value.keys())[:10]}"
            )
        else:
            text = repr(value)
            print(f"{key}: {text[:500]}")

In [ ]:
state = torch.load(
    STATE_DICT_FILE,
    map_location="cpu",
    weights_only=True,
)

if isinstance(state, dict) and "model_state_dict" in state:
    state = state["model_state_dict"]

print("[State dict 파라미터 수]", len(state))
print("\n[마지막 20개 계층]")

for key in list(state.keys())[-20:]:
    value = state[key]

    if hasattr(value, "shape"):
        print(f"{key:50} {tuple(value.shape)}")
    else:
        print(f"{key:50} {type(value)}")

In [ ]:
for filename in ["completed.json", "best_metrics.json"]:
    path = CLASSIFIER_DIR / filename

    print("\n" + "=" * 80)
    print(filename)

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    print(json.dumps(data, ensure_ascii=False, indent=2))

In [ ]:
from pathlib import Path

import torch
import torch.nn as nn
from torchvision.models import resnet18
from torchvision import transforms
from PIL import Image

ROOT = Path("/content/drive/MyDrive/TeamProject/test_dataset")
RUNS = ROOT / "runs"

CLASSIFIER_CHECKPOINT = (
    RUNS
    / "resnet18_dirty3_bboxfixed_pad005"
    / "best_resnet18_dirty3_checkpoint.pt"
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

checkpoint = torch.load(
    CLASSIFIER_CHECKPOINT,
    map_location="cpu",
    weights_only=False,
)

class_names = checkpoint["class_names"]
input_size = checkpoint["input_size"]
normalization_mean = checkpoint["normalization_mean"]
normalization_std = checkpoint["normalization_std"]

classifier = resnet18(weights=None)
classifier.fc = nn.Linear(
    classifier.fc.in_features,
    len(class_names),
)

classifier.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True,
)

classifier = classifier.to(device)
classifier.eval()

classifier_transform = transforms.Compose([
    transforms.Resize((input_size, input_size)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=normalization_mean,
        std=normalization_std,
    ),
])

print("Device:", device)
print("Architecture:", checkpoint["architecture"])
print("Padding:", checkpoint["padding"])
print("Classes:", class_names)
print("Input size:", input_size)
print("Checkpoint epoch:", checkpoint["epoch"])
print("모델 로딩 성공")

In [ ]:
import numpy as np

EXTERNAL_ROOT = ROOT / "실사용테스트"
EXTERNAL_IMAGES = EXTERNAL_ROOT / "images"
EXTERNAL_LABELS = EXTERNAL_ROOT / "labels"

CLASS_9_NAMES = [
    "can_clean",
    "can_outer",
    "can_inner",
    "pet_clean",
    "pet_outer",
    "pet_inner",
    "plastic_clean",
    "plastic_outer",
    "plastic_inner",
]

sample_image_path = (
    EXTERNAL_IMAGES / "can_01_clean_simple.jpg"
)
sample_label_path = (
    EXTERNAL_LABELS / "can_01_clean_simple.txt"
)

image = Image.open(sample_image_path).convert("RGB")
image_width, image_height = image.size

line = sample_label_path.read_text(
    encoding="utf-8"
).strip().splitlines()[0]

parts = line.split()

class_9_id = int(parts[0])
cx, cy, bw, bh = map(float, parts[1:])

x1 = (cx - bw / 2) * image_width
y1 = (cy - bh / 2) * image_height
x2 = (cx + bw / 2) * image_width
y2 = (cy + bh / 2) * image_height

# pad005: bbox 너비와 높이의 5%만큼 사방 확장
pad_x = (x2 - x1) * 0.05
pad_y = (y2 - y1) * 0.05

x1 = max(0, int(round(x1 - pad_x)))
y1 = max(0, int(round(y1 - pad_y)))
x2 = min(image_width, int(round(x2 + pad_x)))
y2 = min(image_height, int(round(y2 + pad_y)))

crop = image.crop((x1, y1, x2, y2))
tensor = classifier_transform(crop).unsqueeze(0).to(device)

with torch.inference_mode():
    logits = classifier(tensor)
    probabilities = torch.softmax(logits, dim=1)[0]
    predicted_dirty_id = int(probabilities.argmax().item())

gt_dirty_id = class_9_id % 3

print("이미지:", sample_image_path.name)
print("GT 9-class:", CLASS_9_NAMES[class_9_id])
print("GT 오염도:", class_names[gt_dirty_id])
print("예측 오염도:", class_names[predicted_dirty_id])

print("\n확률:")
for name, probability in zip(
    class_names,
    probabilities.detach().cpu().tolist(),
):
    print(f"  {name:6}: {probability:.4f}")

In [ ]:
from pathlib import Path
from PIL import Image, ImageEnhance
import shutil
import csv

EXTERNAL_ROOT = ROOT / "실사용테스트"
EXTERNAL_IMAGES = EXTERNAL_ROOT / "images"
EXTERNAL_LABELS = EXTERNAL_ROOT / "labels"

BRIGHTNESS_FACTOR = 0.35

simple_images = sorted(
    EXTERNAL_IMAGES.glob("*_simple.jpg")
)

# 다중 객체는 simple이라는 이름을 사용하지 않지만 안전하게 한 번 더 제외
simple_images = [
    path
    for path in simple_images
    if not path.name.startswith("multi_")
]

print("Simple 이미지 수:", len(simple_images))

if len(simple_images) != 15:
    raise ValueError(
        f"simple 이미지가 15장이 아닙니다: {len(simple_images)}"
    )

manifest_rows = []

for source_image in simple_images:
    dark_image = source_image.with_name(
        source_image.name.replace(
            "_simple.jpg",
            "_dark.jpg",
        )
    )

    source_label = (
        EXTERNAL_LABELS / f"{source_image.stem}.txt"
    )
    dark_label = (
        EXTERNAL_LABELS
        / f"{dark_image.stem}.txt"
    )

    if not source_label.exists():
        raise FileNotFoundError(source_label)

    with Image.open(source_image) as image:
        image = image.convert("RGB")

        darkened = ImageEnhance.Brightness(
            image
        ).enhance(BRIGHTNESS_FACTOR)

        darkened.save(
            dark_image,
            format="JPEG",
            quality=95,
            subsampling=0,
        )

    shutil.copy2(source_label, dark_label)

    manifest_rows.append({
        "source_image": source_image.name,
        "generated_image": dark_image.name,
        "source_label": source_label.name,
        "generated_label": dark_label.name,
        "brightness_factor": BRIGHTNESS_FACTOR,
    })

manifest_path = (
    EXTERNAL_ROOT / "dark_generation_manifest.csv"
)

with open(
    manifest_path,
    "w",
    newline="",
    encoding="utf-8-sig",
) as f:
    writer = csv.DictWriter(
        f,
        fieldnames=manifest_rows[0].keys(),
    )
    writer.writeheader()
    writer.writerows(manifest_rows)

print("저조도 이미지 생성 완료:", len(manifest_rows))
print("Manifest:", manifest_path)

In [ ]:
from collections import Counter

image_files = sorted(
    path
    for path in EXTERNAL_IMAGES.iterdir()
    if path.suffix.lower() in {
        ".jpg", ".jpeg", ".png", ".bmp", ".webp"
    }
)

label_files = sorted(EXTERNAL_LABELS.glob("*.txt"))

image_stems = {path.stem for path in image_files}
label_stems = {path.stem for path in label_files}

condition_counts = Counter()

for image_path in image_files:
    name = image_path.stem

    if name.startswith("multi_"):
        condition = "multi"
    else:
        condition = name.rsplit("_", 1)[-1]

    condition_counts[condition] += 1

gt_objects = 0

for label_path in label_files:
    lines = [
        line
        for line in label_path.read_text(
            encoding="utf-8",
            errors="replace",
        ).splitlines()
        if line.strip()
    ]
    gt_objects += len(lines)

print("Images:", len(image_files))
print("Labels:", len(label_files))
print("GT objects:", gt_objects)
print("Conditions:", dict(sorted(condition_counts.items())))
print("Missing labels:", sorted(image_stems - label_stems))
print("Missing images:", sorted(label_stems - image_stems))

In [ ]:
from pathlib import Path
from PIL import Image

import json
import shutil
import time
import numpy as np
import torch
import torch.nn as nn

from torchvision.models import resnet18
from torchvision import transforms
from ultralytics import YOLO

ROOT = Path("/content/drive/MyDrive/TeamProject/test_dataset")
RUNS = ROOT / "runs"

EXTERNAL_ROOT = ROOT / "실사용테스트"
DRIVE_IMAGES = EXTERNAL_ROOT / "images"
DRIVE_LABELS = EXTERNAL_ROOT / "labels"

ONE_STAGE_WEIGHT = (
    RUNS
    / "yolov8n_1stage_9class_bboxfixed"
    / "weights"
    / "best.pt"
)

TWO_STAGE_WEIGHT = (
    RUNS
    / "yolov8n_2stage_material3_bboxfixed"
    / "weights"
    / "best.pt"
)

CLASSIFIER_CHECKPOINT = (
    RUNS
    / "resnet18_dirty3_bboxfixed_pad005"
    / "best_resnet18_dirty3_checkpoint.pt"
)

RESULT_DIR = RUNS / "external_test_evaluation"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_ROOT = Path("/content/external_test")
LOCAL_IMAGES = LOCAL_ROOT / "images"
LOCAL_LABELS = LOCAL_ROOT / "labels"

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

CLASS_9_NAMES = [
    "can_clean",
    "can_outer",
    "can_inner",
    "pet_clean",
    "pet_outer",
    "pet_inner",
    "plastic_clean",
    "plastic_outer",
    "plastic_inner",
]

MATERIAL_NAMES = ["can", "pet", "plastic"]
DIRTY_NAMES = ["clean", "outer", "inner"]

print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

for path in [
    DRIVE_IMAGES,
    DRIVE_LABELS,
    ONE_STAGE_WEIGHT,
    TWO_STAGE_WEIGHT,
    CLASSIFIER_CHECKPOINT,
]:
    if not path.exists():
        raise FileNotFoundError(path)

print("모든 경로 정상")

In [ ]:
if LOCAL_ROOT.exists():
    shutil.rmtree(LOCAL_ROOT)

LOCAL_IMAGES.mkdir(parents=True, exist_ok=True)
LOCAL_LABELS.mkdir(parents=True, exist_ok=True)

image_extensions = {
    ".jpg", ".jpeg", ".png", ".bmp", ".webp"
}

for source in DRIVE_IMAGES.iterdir():
    if source.is_file() and source.suffix.lower() in image_extensions:
        shutil.copy2(source, LOCAL_IMAGES / source.name)

for source in DRIVE_LABELS.glob("*.txt"):
    shutil.copy2(source, LOCAL_LABELS / source.name)

local_images = sorted(
    path
    for path in LOCAL_IMAGES.iterdir()
    if path.suffix.lower() in image_extensions
)

local_labels = sorted(LOCAL_LABELS.glob("*.txt"))

print("로컬 이미지:", len(local_images))
print("로컬 라벨:", len(local_labels))

if len(local_images) != 67:
    raise ValueError(
        f"이미지가 67장이 아닙니다: {len(local_images)}"
    )

if len(local_labels) != 67:
    raise ValueError(
        f"라벨이 67개가 아닙니다: {len(local_labels)}"
    )

In [ ]:
one_stage_model = YOLO(str(ONE_STAGE_WEIGHT))
two_stage_detector = YOLO(str(TWO_STAGE_WEIGHT))

checkpoint = torch.load(
    CLASSIFIER_CHECKPOINT,
    map_location="cpu",
    weights_only=False,
)

classifier = resnet18(weights=None)
classifier.fc = nn.Linear(
    classifier.fc.in_features,
    len(checkpoint["class_names"]),
)

classifier.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True,
)

classifier = classifier.to(DEVICE)
classifier.eval()

classifier_transform = transforms.Compose([
    transforms.Resize((
        checkpoint["input_size"],
        checkpoint["input_size"],
    )),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=checkpoint["normalization_mean"],
        std=checkpoint["normalization_std"],
    ),
])

print("1-stage names:", one_stage_model.names)
print("2-stage names:", two_stage_detector.names)
print("Classifier names:", checkpoint["class_names"])

expected_one_names = {
    index: name
    for index, name in enumerate(CLASS_9_NAMES)
}

expected_two_names = {
    index: name
    for index, name in enumerate(MATERIAL_NAMES)
}

if one_stage_model.names != expected_one_names:
    raise ValueError(
        f"1-stage 클래스 불일치: {one_stage_model.names}"
    )

if two_stage_detector.names != expected_two_names:
    raise ValueError(
        f"2-stage 클래스 불일치: {two_stage_detector.names}"
    )

if checkpoint["class_names"] != DIRTY_NAMES:
    raise ValueError(
        f"Classifier 클래스 불일치: "
        f"{checkpoint['class_names']}"
    )

print("모든 클래스 매핑 정상")

In [ ]:
def load_yolo_ground_truth(image_path, label_path):
    with Image.open(image_path) as image:
        width, height = image.size

    ground_truths = []

    lines = [
        line.strip()
        for line in label_path.read_text(
            encoding="utf-8",
            errors="replace",
        ).splitlines()
        if line.strip()
    ]

    for object_index, line in enumerate(lines):
        class_id, cx, cy, bw, bh = map(
            float,
            line.split(),
        )

        class_id = int(class_id)

        x1 = (cx - bw / 2) * width
        y1 = (cy - bh / 2) * height
        x2 = (cx + bw / 2) * width
        y2 = (cy + bh / 2) * height

        ground_truths.append({
            "object_index": object_index,
            "class_id": class_id,
            "class_name": CLASS_9_NAMES[class_id],
            "material_id": class_id // 3,
            "material_name": MATERIAL_NAMES[class_id // 3],
            "dirty_id": class_id % 3,
            "dirty_name": DIRTY_NAMES[class_id % 3],
            "bbox": [
                float(x1),
                float(y1),
                float(x2),
                float(y2),
            ],
        })

    return ground_truths


def crop_with_padding(image, bbox, padding_ratio=0.05):
    image_width, image_height = image.size
    x1, y1, x2, y2 = bbox

    box_width = x2 - x1
    box_height = y2 - y1

    pad_x = box_width * padding_ratio
    pad_y = box_height * padding_ratio

    crop_x1 = max(0, int(round(x1 - pad_x)))
    crop_y1 = max(0, int(round(y1 - pad_y)))
    crop_x2 = min(
        image_width,
        int(round(x2 + pad_x)),
    )
    crop_y2 = min(
        image_height,
        int(round(y2 + pad_y)),
    )

    if crop_x2 <= crop_x1 or crop_y2 <= crop_y1:
        return None

    return image.crop(
        (crop_x1, crop_y1, crop_x2, crop_y2)
    )


def predict_dirty_crops(crops):
    if not crops:
        return []

    tensors = torch.stack([
        classifier_transform(crop)
        for crop in crops
    ]).to(DEVICE)

    with torch.inference_mode():
        logits = classifier(tensors)
        probabilities = torch.softmax(
            logits,
            dim=1,
        )

    predictions = []

    for probability in probabilities.detach().cpu():
        dirty_id = int(probability.argmax().item())

        predictions.append({
            "dirty_id": dirty_id,
            "dirty_name": DIRTY_NAMES[dirty_id],
            "dirty_confidence": float(
                probability[dirty_id].item()
            ),
            "dirty_probabilities": {
                DIRTY_NAMES[index]: float(value)
                for index, value in enumerate(
                    probability.tolist()
                )
            },
        })

    return predictions


def synchronize_cuda():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def save_json_atomic(data, destination):
    destination = Path(destination)
    temporary = destination.with_suffix(
        destination.suffix + ".tmp"
    )

    with open(
        temporary,
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            data,
            f,
            ensure_ascii=False,
            indent=2,
        )

    temporary.replace(destination)


def load_cache(path):
    path = Path(path)

    if not path.exists():
        return {}

    with open(path, "r", encoding="utf-8") as f:
        stored = json.load(f)

    return {
        item["image_name"]: item
        for item in stored
    }

In [ ]:
ONE_RAW_PATH = RESULT_DIR / "raw_external_1stage.json"

one_cache = load_cache(ONE_RAW_PATH)

print(
    f"기존 1-stage 캐시: "
    f"{len(one_cache)}/{len(local_images)}"
)

processed_since_save = 0

for image_index, image_path in enumerate(
    local_images,
    start=1,
):
    if image_path.name in one_cache:
        print(
            f"[1-stage {image_index:02}/{len(local_images)}] "
            f"SKIP {image_path.name}"
        )
        continue

    synchronize_cuda()
    started = time.perf_counter()

    result = one_stage_model.predict(
        source=str(image_path),
        imgsz=640,
        conf=0.01,
        iou=0.70,
        max_det=300,
        device=0,
        verbose=False,
    )[0]

    synchronize_cuda()
    elapsed_ms = (
        time.perf_counter() - started
    ) * 1000

    predictions = []

    if result.boxes is not None:
        boxes = result.boxes.xyxy.detach().cpu().numpy()
        confidences = (
            result.boxes.conf.detach().cpu().numpy()
        )
        classes = (
            result.boxes.cls.detach().cpu().numpy()
            .astype(int)
        )

        for bbox, confidence, class_id in zip(
            boxes,
            confidences,
            classes,
        ):
            predictions.append({
                "bbox": [
                    float(value)
                    for value in bbox.tolist()
                ],
                "confidence": float(confidence),
                "class_id": int(class_id),
                "class_name": CLASS_9_NAMES[class_id],
                "material_id": int(class_id // 3),
                "material_name": MATERIAL_NAMES[
                    class_id // 3
                ],
                "dirty_id": int(class_id % 3),
                "dirty_name": DIRTY_NAMES[class_id % 3],
            })

    one_cache[image_path.name] = {
        "image_name": image_path.name,
        "elapsed_ms": float(elapsed_ms),
        "predictions": predictions,
    }

    processed_since_save += 1

    print(
        f"[1-stage {image_index:02}/{len(local_images)}] "
        f"{image_path.name} | "
        f"pred={len(predictions)} | "
        f"{elapsed_ms:.1f}ms"
    )

    if processed_since_save >= 5:
        save_json_atomic(
            list(one_cache.values()),
            ONE_RAW_PATH,
        )
        processed_since_save = 0
        print("  → 중간 저장 완료")

save_json_atomic(
    list(one_cache.values()),
    ONE_RAW_PATH,
)

print(
    "1-stage 완료:",
    len(one_cache),
    ONE_RAW_PATH,
)

In [ ]:
TWO_RAW_PATH = RESULT_DIR / "raw_external_2stage.json"

two_cache = load_cache(TWO_RAW_PATH)

print(
    f"기존 2-stage 캐시: "
    f"{len(two_cache)}/{len(local_images)}"
)

processed_since_save = 0

for image_index, image_path in enumerate(
    local_images,
    start=1,
):
    if image_path.name in two_cache:
        print(
            f"[2-stage {image_index:02}/{len(local_images)}] "
            f"SKIP {image_path.name}"
        )
        continue

    image = Image.open(image_path).convert("RGB")

    synchronize_cuda()
    started = time.perf_counter()

    result = two_stage_detector.predict(
        source=str(image_path),
        imgsz=640,
        conf=0.01,
        iou=0.70,
        max_det=300,
        device=0,
        verbose=False,
    )[0]

    detector_predictions = []
    crops = []

    if result.boxes is not None:
        boxes = result.boxes.xyxy.detach().cpu().numpy()
        confidences = (
            result.boxes.conf.detach().cpu().numpy()
        )
        classes = (
            result.boxes.cls.detach().cpu().numpy()
            .astype(int)
        )

        for bbox, confidence, material_id in zip(
            boxes,
            confidences,
            classes,
        ):
            bbox_list = [
                float(value)
                for value in bbox.tolist()
            ]

            crop = crop_with_padding(
                image,
                bbox_list,
                padding_ratio=0.05,
            )

            if crop is None:
                continue

            detector_predictions.append({
                "bbox": bbox_list,
                "confidence": float(confidence),
                "material_id": int(material_id),
                "material_name": MATERIAL_NAMES[
                    material_id
                ],
            })

            crops.append(crop)

    dirty_predictions = predict_dirty_crops(crops)

    predictions = []

    for detector_prediction, dirty_prediction in zip(
        detector_predictions,
        dirty_predictions,
    ):
        material_id = detector_prediction["material_id"]
        dirty_id = dirty_prediction["dirty_id"]
        final_class_id = material_id * 3 + dirty_id

        predictions.append({
            **detector_prediction,
            **dirty_prediction,
            "class_id": int(final_class_id),
            "class_name": CLASS_9_NAMES[final_class_id],
        })

    synchronize_cuda()
    elapsed_ms = (
        time.perf_counter() - started
    ) * 1000

    two_cache[image_path.name] = {
        "image_name": image_path.name,
        "elapsed_ms": float(elapsed_ms),
        "predictions": predictions,
    }

    image.close()

    processed_since_save += 1

    print(
        f"[2-stage {image_index:02}/{len(local_images)}] "
        f"{image_path.name} | "
        f"pred={len(predictions)} | "
        f"{elapsed_ms:.1f}ms"
    )

    if processed_since_save >= 5:
        save_json_atomic(
            list(two_cache.values()),
            TWO_RAW_PATH,
        )
        processed_since_save = 0
        print("  → 중간 저장 완료")

save_json_atomic(
    list(two_cache.values()),
    TWO_RAW_PATH,
)

print(
    "2-stage 완료:",
    len(two_cache),
    TWO_RAW_PATH,
)

In [ ]:
ORACLE_RAW_PATH = (
    RESULT_DIR / "raw_external_oracle_classifier.json"
)

oracle_cache = load_cache(ORACLE_RAW_PATH)

print(
    f"기존 Oracle 캐시: "
    f"{len(oracle_cache)}/{len(local_images)}"
)

processed_since_save = 0

for image_index, image_path in enumerate(
    local_images,
    start=1,
):
    if image_path.name in oracle_cache:
        print(
            f"[Oracle {image_index:02}/{len(local_images)}] "
            f"SKIP {image_path.name}"
        )
        continue

    label_path = (
        LOCAL_LABELS / f"{image_path.stem}.txt"
    )

    ground_truths = load_yolo_ground_truth(
        image_path,
        label_path,
    )

    image = Image.open(image_path).convert("RGB")

    crops = [
        crop_with_padding(
            image,
            ground_truth["bbox"],
            padding_ratio=0.05,
        )
        for ground_truth in ground_truths
    ]

    if any(crop is None for crop in crops):
        raise ValueError(
            f"잘못된 GT crop: {image_path.name}"
        )

    synchronize_cuda()
    started = time.perf_counter()

    dirty_predictions = predict_dirty_crops(crops)

    synchronize_cuda()
    elapsed_ms = (
        time.perf_counter() - started
    ) * 1000

    predictions = []

    for ground_truth, dirty_prediction in zip(
        ground_truths,
        dirty_predictions,
    ):
        material_id = ground_truth["material_id"]
        dirty_id = dirty_prediction["dirty_id"]
        final_class_id = material_id * 3 + dirty_id

        predictions.append({
            "object_index": ground_truth["object_index"],
            "gt_bbox": ground_truth["bbox"],
            "gt_class_id": ground_truth["class_id"],
            "gt_class_name": ground_truth["class_name"],
            "gt_material_id": material_id,
            "gt_dirty_id": ground_truth["dirty_id"],
            **dirty_prediction,
            "class_id": int(final_class_id),
            "class_name": CLASS_9_NAMES[final_class_id],
        })

    oracle_cache[image_path.name] = {
        "image_name": image_path.name,
        "elapsed_ms": float(elapsed_ms),
        "predictions": predictions,
    }

    image.close()

    processed_since_save += 1

    print(
        f"[Oracle {image_index:02}/{len(local_images)}] "
        f"{image_path.name} | "
        f"objects={len(predictions)} | "
        f"{elapsed_ms:.1f}ms"
    )

    if processed_since_save >= 5:
        save_json_atomic(
            list(oracle_cache.values()),
            ORACLE_RAW_PATH,
        )
        processed_since_save = 0
        print("  → 중간 저장 완료")

save_json_atomic(
    list(oracle_cache.values()),
    ORACLE_RAW_PATH,
)

print(
    "Oracle 완료:",
    len(oracle_cache),
    ORACLE_RAW_PATH,
)

In [ ]:
for name, path in {
    "1-stage": ONE_RAW_PATH,
    "2-stage": TWO_RAW_PATH,
    "Oracle": ORACLE_RAW_PATH,
}.items():
    with open(path, "r", encoding="utf-8") as f:
        records = json.load(f)

    prediction_count = sum(
        len(record["predictions"])
        for record in records
    )

    print(
        f"{name:10} | "
        f"images={len(records)} | "
        f"predictions={prediction_count} | "
        f"file={path.name}"
    )

In [ ]:
from pathlib import Path
from PIL import Image
from scipy.optimize import linear_sum_assignment
from collections import defaultdict

import json
import numpy as np
import pandas as pd

ROOT = Path("/content/drive/MyDrive/TeamProject/test_dataset")
RUNS = ROOT / "runs"
RESULT_DIR = RUNS / "external_test_evaluation"

EXTERNAL_ROOT = ROOT / "실사용테스트"
EXTERNAL_IMAGES = EXTERNAL_ROOT / "images"
EXTERNAL_LABELS = EXTERNAL_ROOT / "labels"

ONE_RAW_PATH = RESULT_DIR / "raw_external_1stage.json"
TWO_RAW_PATH = RESULT_DIR / "raw_external_2stage.json"
ORACLE_RAW_PATH = (
    RESULT_DIR / "raw_external_oracle_classifier.json"
)

CLASS_9_NAMES = [
    "can_clean",
    "can_outer",
    "can_inner",
    "pet_clean",
    "pet_outer",
    "pet_inner",
    "plastic_clean",
    "plastic_outer",
    "plastic_inner",
]

MATERIAL_NAMES = ["can", "pet", "plastic"]
DIRTY_NAMES = ["clean", "outer", "inner"]

IOU_THRESHOLD = 0.5
PRIMARY_CONFIDENCE = 0.5
SECONDARY_CONFIDENCE = 0.25

In [ ]:
from pathlib import Path
from PIL import Image
from scipy.optimize import linear_sum_assignment
from collections import defaultdict

import json
import numpy as np
import pandas as pd

ROOT = Path("/content/drive/MyDrive/TeamProject/test_dataset")
RUNS = ROOT / "runs"
RESULT_DIR = RUNS / "external_test_evaluation"

EXTERNAL_ROOT = ROOT / "실사용테스트"
EXTERNAL_IMAGES = EXTERNAL_ROOT / "images"
EXTERNAL_LABELS = EXTERNAL_ROOT / "labels"

ONE_RAW_PATH = RESULT_DIR / "raw_external_1stage.json"
TWO_RAW_PATH = RESULT_DIR / "raw_external_2stage.json"
ORACLE_RAW_PATH = (
    RESULT_DIR / "raw_external_oracle_classifier.json"
)

CLASS_9_NAMES = [
    "can_clean",
    "can_outer",
    "can_inner",
    "pet_clean",
    "pet_outer",
    "pet_inner",
    "plastic_clean",
    "plastic_outer",
    "plastic_inner",
]

MATERIAL_NAMES = ["can", "pet", "plastic"]
DIRTY_NAMES = ["clean", "outer", "inner"]

IOU_THRESHOLD = 0.5
PRIMARY_CONFIDENCE = 0.5
SECONDARY_CONFIDENCE = 0.25

In [ ]:
def load_json_records(path):
    with open(path, "r", encoding="utf-8") as f:
        records = json.load(f)

    return {
        record["image_name"]: record
        for record in records
    }


def get_condition(image_name):
    stem = Path(image_name).stem

    if stem.startswith("multi_"):
        return "multi"

    return stem.rsplit("_", 1)[-1]


def load_gt(image_name):
    image_path = EXTERNAL_IMAGES / image_name
    label_path = (
        EXTERNAL_LABELS
        / f"{Path(image_name).stem}.txt"
    )

    with Image.open(image_path) as image:
        width, height = image.size

    ground_truths = []

    lines = [
        line.strip()
        for line in label_path.read_text(
            encoding="utf-8",
            errors="replace",
        ).splitlines()
        if line.strip()
    ]

    for object_index, line in enumerate(lines):
        class_id, cx, cy, bw, bh = map(
            float,
            line.split(),
        )
        class_id = int(class_id)

        x1 = (cx - bw / 2) * width
        y1 = (cy - bh / 2) * height
        x2 = (cx + bw / 2) * width
        y2 = (cy + bh / 2) * height

        ground_truths.append({
            "object_index": object_index,
            "class_id": class_id,
            "class_name": CLASS_9_NAMES[class_id],
            "material_id": class_id // 3,
            "material_name": MATERIAL_NAMES[class_id // 3],
            "dirty_id": class_id % 3,
            "dirty_name": DIRTY_NAMES[class_id % 3],
            "bbox": [
                float(x1),
                float(y1),
                float(x2),
                float(y2),
            ],
        })

    return ground_truths


def intersection_area(box_a, box_b):
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b

    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)

    return max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)


def box_area(box):
    x1, y1, x2, y2 = box
    return max(0.0, x2 - x1) * max(0.0, y2 - y1)


def calculate_iou(box_a, box_b):
    intersection = intersection_area(box_a, box_b)

    union = (
        box_area(box_a)
        + box_area(box_b)
        - intersection
    )

    if union <= 0:
        return 0.0

    return intersection / union


def build_iou_matrix(ground_truths, predictions):
    matrix = np.zeros(
        (len(ground_truths), len(predictions)),
        dtype=np.float64,
    )

    for gt_index, ground_truth in enumerate(
        ground_truths
    ):
        for pred_index, prediction in enumerate(
            predictions
        ):
            matrix[gt_index, pred_index] = (
                calculate_iou(
                    ground_truth["bbox"],
                    prediction["bbox"],
                )
            )

    return matrix


def match_one_to_one(
    ground_truths,
    predictions,
    iou_threshold=0.5,
):
    if not ground_truths or not predictions:
        return {}, set(range(len(predictions)))

    iou_matrix = build_iou_matrix(
        ground_truths,
        predictions,
    )

    gt_indices, pred_indices = linear_sum_assignment(
        -iou_matrix
    )

    matches = {}
    matched_prediction_indices = set()

    for gt_index, pred_index in zip(
        gt_indices,
        pred_indices,
    ):
        iou = float(iou_matrix[gt_index, pred_index])

        if iou >= iou_threshold:
            matches[int(gt_index)] = {
                "pred_index": int(pred_index),
                "iou": iou,
            }
            matched_prediction_indices.add(
                int(pred_index)
            )

    unmatched_predictions = (
        set(range(len(predictions)))
        - matched_prediction_indices
    )

    return matches, unmatched_predictions

In [ ]:
def evaluate_detection_pipeline(
    raw_path,
    pipeline,
    confidence_threshold,
    iou_threshold=0.5,
):
    raw_records = load_json_records(raw_path)

    detail_rows = []
    false_prediction_rows = []
    image_rows = []

    for image_name in sorted(raw_records):
        condition = get_condition(image_name)
        ground_truths = load_gt(image_name)

        predictions = [
            prediction
            for prediction
            in raw_records[image_name]["predictions"]
            if prediction["confidence"]
            >= confidence_threshold
        ]

        matches, unmatched_prediction_indices = (
            match_one_to_one(
                ground_truths,
                predictions,
                iou_threshold=iou_threshold,
            )
        )

        # 하나의 예측 박스가 GT 두 개 이상을 상당 부분 덮는지 검사
        possible_merge_count = 0

        for prediction in predictions:
            covered_gt_count = 0

            for ground_truth in ground_truths:
                gt_area = box_area(
                    ground_truth["bbox"]
                )

                coverage = (
                    intersection_area(
                        prediction["bbox"],
                        ground_truth["bbox"],
                    ) / gt_area
                    if gt_area > 0
                    else 0.0
                )

                if coverage >= 0.5:
                    covered_gt_count += 1

            if covered_gt_count >= 2:
                possible_merge_count += 1

        # 동일 GT에 IoU 0.5 이상 예측이 여러 개 붙는지 검사
        possible_duplicate_count = 0

        for ground_truth in ground_truths:
            overlapping_predictions = sum(
                calculate_iou(
                    ground_truth["bbox"],
                    prediction["bbox"],
                ) >= iou_threshold
                for prediction in predictions
            )

            if overlapping_predictions >= 2:
                possible_duplicate_count += (
                    overlapping_predictions - 1
                )

        for gt_index, ground_truth in enumerate(
            ground_truths
        ):
            match = matches.get(gt_index)

            if match is None:
                prediction = None
                matched = False
                matched_iou = np.nan
                confidence = np.nan
                pred_class_id = np.nan
                pred_class_name = None
                pred_material_id = np.nan
                pred_material_name = None
                pred_dirty_id = np.nan
                pred_dirty_name = None
            else:
                prediction = predictions[
                    match["pred_index"]
                ]
                matched = True
                matched_iou = match["iou"]
                confidence = prediction["confidence"]
                pred_class_id = prediction["class_id"]
                pred_class_name = prediction["class_name"]
                pred_material_id = prediction[
                    "material_id"
                ]
                pred_material_name = prediction[
                    "material_name"
                ]
                pred_dirty_id = prediction["dirty_id"]
                pred_dirty_name = prediction[
                    "dirty_name"
                ]

            material_correct = bool(
                matched
                and pred_material_id
                == ground_truth["material_id"]
            )

            dirty_correct = bool(
                matched
                and pred_dirty_id
                == ground_truth["dirty_id"]
            )

            exact_correct = bool(
                matched
                and pred_class_id
                == ground_truth["class_id"]
            )

            detail_rows.append({
                "pipeline": pipeline,
                "confidence_threshold": (
                    confidence_threshold
                ),
                "iou_threshold": iou_threshold,
                "image_name": image_name,
                "condition": condition,
                "object_index": (
                    ground_truth["object_index"]
                ),
                "gt_class_id": (
                    ground_truth["class_id"]
                ),
                "gt_class_name": (
                    ground_truth["class_name"]
                ),
                "gt_material_id": (
                    ground_truth["material_id"]
                ),
                "gt_material_name": (
                    ground_truth["material_name"]
                ),
                "gt_dirty_id": (
                    ground_truth["dirty_id"]
                ),
                "gt_dirty_name": (
                    ground_truth["dirty_name"]
                ),
                "matched": matched,
                "matched_iou": matched_iou,
                "pred_confidence": confidence,
                "pred_class_id": pred_class_id,
                "pred_class_name": pred_class_name,
                "pred_material_id": pred_material_id,
                "pred_material_name": (
                    pred_material_name
                ),
                "pred_dirty_id": pred_dirty_id,
                "pred_dirty_name": pred_dirty_name,
                "material_correct": material_correct,
                "dirty_correct": dirty_correct,
                "exact_correct": exact_correct,
            })

        for pred_index in sorted(
            unmatched_prediction_indices
        ):
            prediction = predictions[pred_index]

            false_prediction_rows.append({
                "pipeline": pipeline,
                "confidence_threshold": (
                    confidence_threshold
                ),
                "image_name": image_name,
                "condition": condition,
                "pred_index": pred_index,
                "pred_confidence": (
                    prediction["confidence"]
                ),
                "pred_class_id": prediction["class_id"],
                "pred_class_name": (
                    prediction["class_name"]
                ),
                "pred_material_id": (
                    prediction["material_id"]
                ),
                "pred_material_name": (
                    prediction["material_name"]
                ),
                "pred_dirty_id": (
                    prediction["dirty_id"]
                ),
                "pred_dirty_name": (
                    prediction["dirty_name"]
                ),
                "bbox": json.dumps(
                    prediction["bbox"]
                ),
            })

        image_rows.append({
            "pipeline": pipeline,
            "confidence_threshold": (
                confidence_threshold
            ),
            "image_name": image_name,
            "condition": condition,
            "gt_count": len(ground_truths),
            "prediction_count": len(predictions),
            "matched_count": len(matches),
            "missed_gt_count": (
                len(ground_truths) - len(matches)
            ),
            "false_prediction_count": len(
                unmatched_prediction_indices
            ),
            "possible_merge_count": (
                possible_merge_count
            ),
            "possible_duplicate_count": (
                possible_duplicate_count
            ),
            "elapsed_ms": raw_records[
                image_name
            ]["elapsed_ms"],
        })

    return (
        pd.DataFrame(detail_rows),
        pd.DataFrame(false_prediction_rows),
        pd.DataFrame(image_rows),
    )

In [ ]:
evaluation_results = {}

for confidence_threshold in [
    PRIMARY_CONFIDENCE,
    SECONDARY_CONFIDENCE,
]:
    for pipeline, raw_path in [
        ("1-stage", ONE_RAW_PATH),
        ("2-stage", TWO_RAW_PATH),
    ]:
        print(
            f"평가 중: {pipeline}, "
            f"conf={confidence_threshold}"
        )

        evaluation_results[
            (pipeline, confidence_threshold)
        ] = evaluate_detection_pipeline(
            raw_path=raw_path,
            pipeline=pipeline,
            confidence_threshold=(
                confidence_threshold
            ),
            iou_threshold=IOU_THRESHOLD,
        )

print("Detection pipeline 평가 완료")

In [ ]:
def calculate_exact_macro_f1(
    detail_subset,
    false_prediction_subset,
):
    present_classes = sorted(
        detail_subset["gt_class_id"]
        .astype(int)
        .unique()
        .tolist()
    )

    class_rows = []

    for class_id in present_classes:
        true_positive = 0
        false_positive = 0
        false_negative = 0

        for _, row in detail_subset.iterrows():
            gt_class_id = int(row["gt_class_id"])

            if bool(row["matched"]):
                pred_class_id = int(
                    row["pred_class_id"]
                )

                if (
                    gt_class_id == class_id
                    and pred_class_id == class_id
                ):
                    true_positive += 1
                else:
                    if gt_class_id == class_id:
                        false_negative += 1
                    if pred_class_id == class_id:
                        false_positive += 1
            elif gt_class_id == class_id:
                false_negative += 1

        if not false_prediction_subset.empty:
            false_positive += int(
                (
                    false_prediction_subset[
                        "pred_class_id"
                    ].astype(int)
                    == class_id
                ).sum()
            )

        precision = (
            true_positive
            / (true_positive + false_positive)
            if true_positive + false_positive > 0
            else 0.0
        )

        recall = (
            true_positive
            / (true_positive + false_negative)
            if true_positive + false_negative > 0
            else 0.0
        )

        f1 = (
            2 * precision * recall
            / (precision + recall)
            if precision + recall > 0
            else 0.0
        )

        class_rows.append({
            "class_id": class_id,
            "class_name": CLASS_9_NAMES[class_id],
            "support": int(
                (
                    detail_subset["gt_class_id"]
                    .astype(int)
                    == class_id
                ).sum()
            ),
            "tp": true_positive,
            "fp": false_positive,
            "fn": false_negative,
            "precision": precision,
            "recall": recall,
            "f1": f1,
        })

    macro_f1 = (
        float(np.mean([
            row["f1"]
            for row in class_rows
        ]))
        if class_rows
        else np.nan
    )

    return macro_f1, class_rows

In [ ]:
def calculate_exact_macro_f1(
    detail_subset,
    false_prediction_subset,
):
    present_classes = sorted(
        detail_subset["gt_class_id"]
        .astype(int)
        .unique()
        .tolist()
    )

    class_rows = []

    for class_id in present_classes:
        true_positive = 0
        false_positive = 0
        false_negative = 0

        for _, row in detail_subset.iterrows():
            gt_class_id = int(row["gt_class_id"])

            if bool(row["matched"]):
                pred_class_id = int(
                    row["pred_class_id"]
                )

                if (
                    gt_class_id == class_id
                    and pred_class_id == class_id
                ):
                    true_positive += 1
                else:
                    if gt_class_id == class_id:
                        false_negative += 1
                    if pred_class_id == class_id:
                        false_positive += 1
            elif gt_class_id == class_id:
                false_negative += 1

        if not false_prediction_subset.empty:
            false_positive += int(
                (
                    false_prediction_subset[
                        "pred_class_id"
                    ].astype(int)
                    == class_id
                ).sum()
            )

        precision = (
            true_positive
            / (true_positive + false_positive)
            if true_positive + false_positive > 0
            else 0.0
        )

        recall = (
            true_positive
            / (true_positive + false_negative)
            if true_positive + false_negative > 0
            else 0.0
        )

        f1 = (
            2 * precision * recall
            / (precision + recall)
            if precision + recall > 0
            else 0.0
        )

        class_rows.append({
            "class_id": class_id,
            "class_name": CLASS_9_NAMES[class_id],
            "support": int(
                (
                    detail_subset["gt_class_id"]
                    .astype(int)
                    == class_id
                ).sum()
            ),
            "tp": true_positive,
            "fp": false_positive,
            "fn": false_negative,
            "precision": precision,
            "recall": recall,
            "f1": f1,
        })

    macro_f1 = (
        float(np.mean([
            row["f1"]
            for row in class_rows
        ]))
        if class_rows
        else np.nan
    )

    return macro_f1, class_rows

In [ ]:
def build_summaries(
    detail_df,
    false_df,
    image_df,
):
    summary_rows = []
    per_class_rows = []

    conditions = [
        "all",
        *sorted(detail_df["condition"].unique()),
    ]

    for condition in conditions:
        if condition == "all":
            details = detail_df.copy()
            images = image_df.copy()
            false_predictions = false_df.copy()
        else:
            details = detail_df[
                detail_df["condition"] == condition
            ].copy()

            images = image_df[
                image_df["condition"] == condition
            ].copy()

            false_predictions = false_df[
                false_df["condition"] == condition
            ].copy() if not false_df.empty else false_df.copy()

        gt_count = len(details)
        prediction_count = int(
            images["prediction_count"].sum()
        )
        matched_count = int(
            details["matched"].sum()
        )
        false_count = int(
            images["false_prediction_count"].sum()
        )

        macro_f1, class_rows = (
            calculate_exact_macro_f1(
                details,
                false_predictions,
            )
        )

        for row in class_rows:
            per_class_rows.append({
                "pipeline": details[
                    "pipeline"
                ].iloc[0],
                "confidence_threshold": float(
                    details[
                        "confidence_threshold"
                    ].iloc[0]
                ),
                "condition": condition,
                **row,
            })

        summary_rows.append({
            "pipeline": details["pipeline"].iloc[0],
            "confidence_threshold": float(
                details[
                    "confidence_threshold"
                ].iloc[0]
            ),
            "iou_threshold": float(
                details["iou_threshold"].iloc[0]
            ),
            "condition": condition,
            "images": int(
                images["image_name"].nunique()
            ),
            "gt_objects": gt_count,
            "predictions": prediction_count,
            "localized_gt": matched_count,
            "localization_precision": (
                matched_count / prediction_count
                if prediction_count > 0
                else 0.0
            ),
            "localization_recall": (
                matched_count / gt_count
                if gt_count > 0
                else 0.0
            ),
            "false_predictions": false_count,
            "missed_gt": gt_count - matched_count,
            "material_accuracy_all_gt": float(
                details["material_correct"].mean()
            ),
            "dirty_accuracy_all_gt": float(
                details["dirty_correct"].mean()
            ),
            "exact_9class_accuracy_all_gt": float(
                details["exact_correct"].mean()
            ),
            "exact_9class_accuracy_localized": (
                float(
                    details.loc[
                        details["matched"],
                        "exact_correct",
                    ].mean()
                )
                if matched_count > 0
                else np.nan
            ),
            "exact_9class_macro_f1_present_classes": (
                macro_f1
            ),
            "mean_matched_iou": (
                float(
                    details.loc[
                        details["matched"],
                        "matched_iou",
                    ].mean()
                )
                if matched_count > 0
                else np.nan
            ),
            "possible_merges": int(
                images["possible_merge_count"].sum()
            ),
            "possible_duplicates": int(
                images[
                    "possible_duplicate_count"
                ].sum()
            ),
            "mean_wall_ms_per_image": float(
                images["elapsed_ms"].mean()
            ),
            "median_wall_ms_per_image": float(
                images["elapsed_ms"].median()
            ),
            "p90_wall_ms_per_image": float(
                images["elapsed_ms"].quantile(0.90)
            ),
        })

    return (
        pd.DataFrame(summary_rows),
        pd.DataFrame(per_class_rows),
    )

In [ ]:
oracle_records = load_json_records(
    ORACLE_RAW_PATH
)

oracle_rows = []

for image_name in sorted(oracle_records):
    condition = get_condition(image_name)

    for prediction in oracle_records[
        image_name
    ]["predictions"]:
        gt_class_id = int(
            prediction["gt_class_id"]
        )
        pred_class_id = int(
            prediction["class_id"]
        )

        oracle_rows.append({
            "pipeline": "2-stage Oracle",
            "image_name": image_name,
            "condition": condition,
            "object_index": int(
                prediction["object_index"]
            ),
            "gt_class_id": gt_class_id,
            "gt_class_name": (
                CLASS_9_NAMES[gt_class_id]
            ),
            "gt_material_id": int(
                prediction["gt_material_id"]
            ),
            "gt_dirty_id": int(
                prediction["gt_dirty_id"]
            ),
            "pred_dirty_id": int(
                prediction["dirty_id"]
            ),
            "pred_dirty_name": (
                prediction["dirty_name"]
            ),
            "pred_dirty_confidence": float(
                prediction["dirty_confidence"]
            ),
            "pred_class_id": pred_class_id,
            "pred_class_name": (
                CLASS_9_NAMES[pred_class_id]
            ),
            "dirty_correct": (
                int(prediction["dirty_id"])
                == int(prediction["gt_dirty_id"])
            ),
            "exact_correct": (
                pred_class_id == gt_class_id
            ),
        })

oracle_detail_df = pd.DataFrame(oracle_rows)

oracle_summary_rows = []

for condition in [
    "all",
    *sorted(
        oracle_detail_df["condition"].unique()
    ),
]:
    if condition == "all":
        subset = oracle_detail_df
    else:
        subset = oracle_detail_df[
            oracle_detail_df["condition"]
            == condition
        ]

    present_classes = sorted(
        subset["gt_class_id"]
        .astype(int)
        .unique()
    )

    f1_values = []

    for class_id in present_classes:
        true_positive = int(
            (
                (subset["gt_class_id"] == class_id)
                & (
                    subset["pred_class_id"]
                    == class_id
                )
            ).sum()
        )

        false_positive = int(
            (
                (subset["gt_class_id"] != class_id)
                & (
                    subset["pred_class_id"]
                    == class_id
                )
            ).sum()
        )

        false_negative = int(
            (
                (subset["gt_class_id"] == class_id)
                & (
                    subset["pred_class_id"]
                    != class_id
                )
            ).sum()
        )

        denominator = (
            2 * true_positive
            + false_positive
            + false_negative
        )

        f1_values.append(
            2 * true_positive / denominator
            if denominator > 0
            else 0.0
        )

    oracle_summary_rows.append({
        "pipeline": "2-stage Oracle",
        "condition": condition,
        "images": int(
            subset["image_name"].nunique()
        ),
        "gt_objects": len(subset),
        "dirty_accuracy": float(
            subset["dirty_correct"].mean()
        ),
        "exact_9class_accuracy": float(
            subset["exact_correct"].mean()
        ),
        "macro_f1_present_classes": float(
            np.mean(f1_values)
        ),
    })

oracle_summary_df = pd.DataFrame(
    oracle_summary_rows
)

print("Oracle 계산 완료")

In [ ]:
summary_path = (
    RESULT_DIR
    / "external_summary_all_thresholds.csv"
)

per_class_path = (
    RESULT_DIR
    / "external_per_class_all_thresholds.csv"
)

oracle_summary_path = (
    RESULT_DIR
    / "external_oracle_summary.csv"
)

oracle_detail_path = (
    RESULT_DIR
    / "external_oracle_details.csv"
)

summary_all.to_csv(
    summary_path,
    index=False,
    encoding="utf-8-sig",
)

per_class_all.to_csv(
    per_class_path,
    index=False,
    encoding="utf-8-sig",
)

oracle_summary_df.to_csv(
    oracle_summary_path,
    index=False,
    encoding="utf-8-sig",
)

oracle_detail_df.to_csv(
    oracle_detail_path,
    index=False,
    encoding="utf-8-sig",
)

# confidence 0.5의 상세 결과 저장
for pipeline in ["1-stage", "2-stage"]:
    detail_df, false_df, image_df = (
        evaluation_results[
            (pipeline, PRIMARY_CONFIDENCE)
        ]
    )

    safe_name = pipeline.replace("-", "_")

    detail_df.to_csv(
        RESULT_DIR
        / f"external_{safe_name}_details_conf050.csv",
        index=False,
        encoding="utf-8-sig",
    )

    false_df.to_csv(
        RESULT_DIR
        / f"external_{safe_name}_false_predictions_conf050.csv",
        index=False,
        encoding="utf-8-sig",
    )

    image_df.to_csv(
        RESULT_DIR
        / f"external_{safe_name}_image_diagnostics_conf050.csv",
        index=False,
        encoding="utf-8-sig",
    )

print("저장 완료:", RESULT_DIR)

In [ ]:
variables_to_check = [
    "evaluation_results",
    "build_summaries",
    "summary_all",
    "per_class_all",
    "oracle_summary_df",
    "oracle_detail_df",
]

for variable_name in variables_to_check:
    print(
        f"{variable_name:25}",
        "OK" if variable_name in globals() else "MISSING",
    )

In [ ]:
all_summary_frames = []
all_per_class_frames = []

for (
    pipeline,
    confidence_threshold,
), (
    detail_df,
    false_df,
    image_df,
) in evaluation_results.items():

    summary_df, per_class_df = build_summaries(
        detail_df,
        false_df,
        image_df,
    )

    all_summary_frames.append(summary_df)
    all_per_class_frames.append(per_class_df)

summary_all = pd.concat(
    all_summary_frames,
    ignore_index=True,
)

per_class_all = pd.concat(
    all_per_class_frames,
    ignore_index=True,
)

print("summary_all 생성:", summary_all.shape)
print("per_class_all 생성:", per_class_all.shape)

display(summary_all)

In [ ]:
required_variables = {
    "summary_all": summary_all,
    "per_class_all": per_class_all,
    "oracle_summary_df": oracle_summary_df,
    "oracle_detail_df": oracle_detail_df,
}

for name, value in required_variables.items():
    print(f"{name:25} OK | shape={value.shape}")

In [ ]:
summary_path = (
    RESULT_DIR
    / "external_summary_all_thresholds.csv"
)

per_class_path = (
    RESULT_DIR
    / "external_per_class_all_thresholds.csv"
)

oracle_summary_path = (
    RESULT_DIR
    / "external_oracle_summary.csv"
)

oracle_detail_path = (
    RESULT_DIR
    / "external_oracle_details.csv"
)

summary_all.to_csv(
    summary_path,
    index=False,
    encoding="utf-8-sig",
)

per_class_all.to_csv(
    per_class_path,
    index=False,
    encoding="utf-8-sig",
)

oracle_summary_df.to_csv(
    oracle_summary_path,
    index=False,
    encoding="utf-8-sig",
)

oracle_detail_df.to_csv(
    oracle_detail_path,
    index=False,
    encoding="utf-8-sig",
)

# confidence 0.5의 상세 결과 저장
for pipeline in ["1-stage", "2-stage"]:
    detail_df, false_df, image_df = (
        evaluation_results[
            (pipeline, PRIMARY_CONFIDENCE)
        ]
    )

    safe_name = pipeline.replace("-", "_")

    detail_df.to_csv(
        RESULT_DIR
        / f"external_{safe_name}_details_conf050.csv",
        index=False,
        encoding="utf-8-sig",
    )

    false_df.to_csv(
        RESULT_DIR
        / f"external_{safe_name}_false_predictions_conf050.csv",
        index=False,
        encoding="utf-8-sig",
    )

    image_df.to_csv(
        RESULT_DIR
        / f"external_{safe_name}_image_diagnostics_conf050.csv",
        index=False,
        encoding="utf-8-sig",
    )

print("저장 완료:", RESULT_DIR)

In [ ]:
primary_overall = summary_all[
    (summary_all["confidence_threshold"] == 0.5)
    & (summary_all["condition"] == "all")
]

display(
    primary_overall[[
        "pipeline",
        "images",
        "gt_objects",
        "predictions",
        "localized_gt",
        "localization_precision",
        "localization_recall",
        "false_predictions",
        "missed_gt",
        "material_accuracy_all_gt",
        "dirty_accuracy_all_gt",
        "exact_9class_accuracy_all_gt",
        "exact_9class_macro_f1_present_classes",
        "mean_matched_iou",
        "possible_merges",
        "possible_duplicates",
    ]]
)

In [ ]:
primary_by_condition = summary_all[
    summary_all["confidence_threshold"] == 0.5
].copy()

display(
    primary_by_condition[[
        "pipeline",
        "condition",
        "images",
        "gt_objects",
        "predictions",
        "localization_precision",
        "localization_recall",
        "material_accuracy_all_gt",
        "dirty_accuracy_all_gt",
        "exact_9class_accuracy_all_gt",
        "exact_9class_macro_f1_present_classes",
        "false_predictions",
        "missed_gt",
        "possible_merges",
    ]].sort_values([
        "condition",
        "pipeline",
    ])
)

In [ ]:
display(oracle_summary_df)

In [ ]:
display(
    summary_all[
        summary_all["condition"] == "all"
    ][[
        "pipeline",
        "confidence_threshold",
        "localization_precision",
        "localization_recall",
        "material_accuracy_all_gt",
        "dirty_accuracy_all_gt",
        "exact_9class_accuracy_all_gt",
        "false_predictions",
        "missed_gt",
    ]].sort_values([
        "pipeline",
        "confidence_threshold",
    ])
)

In [ ]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED

ROOT = Path("/content/drive/MyDrive/TeamProject/test_dataset")

EXTERNAL_ROOT = ROOT / "실사용테스트"
IMAGE_DIR = EXTERNAL_ROOT / "images"
LABEL_DIR = EXTERNAL_ROOT / "labels"

RESULT_DIR = (
    ROOT / "runs" / "external_test_evaluation"
)

OUTPUT_ZIP = Path(
    "/content/external_test_results_share.zip"
)

if OUTPUT_ZIP.exists():
    OUTPUT_ZIP.unlink()

with ZipFile(
    OUTPUT_ZIP,
    mode="w",
    compression=ZIP_DEFLATED,
    compresslevel=6,
) as zip_file:

    # 이미지 67장
    for path in sorted(IMAGE_DIR.iterdir()):
        if (
            path.is_file()
            and path.suffix.lower()
            in {".jpg", ".jpeg", ".png", ".webp"}
        ):
            zip_file.write(
                path,
                arcname=f"images/{path.name}",
            )

    # GT 라벨 67개
    for path in sorted(LABEL_DIR.glob("*.txt")):
        zip_file.write(
            path,
            arcname=f"labels/{path.name}",
        )

    # 평가 결과 CSV, JSON, PNG
    for path in sorted(RESULT_DIR.rglob("*")):
        if (
            path.is_file()
            and path.suffix.lower()
            in {".csv", ".json", ".png"}
        ):
            relative = path.relative_to(RESULT_DIR)

            zip_file.write(
                path,
                arcname=f"results/{relative}",
            )

    # 저조도 생성 정보
    manifest = (
        EXTERNAL_ROOT
        / "dark_generation_manifest.csv"
    )

    if manifest.exists():
        zip_file.write(
            manifest,
            arcname="dark_generation_manifest.csv",
        )

print("압축 완료:", OUTPUT_ZIP)
print(
    "크기:",
    f"{OUTPUT_ZIP.stat().st_size / 1024 / 1024:.2f} MB",
)

In [ ]:
from google.colab import files

files.download(
    "/content/external_test_results_share.zip"
)